In [ ]:
# AKI LLM Importance Score Collection - Adjust Task
#
# Queries an LLM to score each feature's clinical relevance for predicting
# postoperative AKI (creatinine ratio at hour 60) in cardiac surgery patients.
# Features are scored in batches using structured JSON outputs
# (Pydantic schema). Results are cached incrementally and exported as a CSV
# compatible with the LSP analysis pipeline.
#
# Requirements: openai, pydantic, pandas, numpy

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import pandas as pd
path = "/content/drive/MyDrive/t60_reg_data.csv"
df = pd.read_csv(path)

In [ ]:
import os
import json
import math
import time
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI



# ------------------------------------------------------------------------------
# Data Loading
# ------------------------------------------------------------------------------

DATA_PATH = "/content/drive/MyDrive/t60_reg_data.csv"
ID_COLS = ["pat_id"]
TARGET_COL = "creatinine_ratio"

df = pd.read_csv(DATA_PATH)

FEATURE_COLS = [c for c in df.columns if c not in ID_COLS + [TARGET_COL]]
print("n_features =", len(FEATURE_COLS))


# ------------------------------------------------------------------------------
# Prompt Construction
# ------------------------------------------------------------------------------

DATASET_BACKGROUND = """
We have adult patients, 80 years and older, who underwent cardiac surgery.
For each patient, we have perioperative and ICU EMR-derived features:
demographics, comorbidities, vitals/hemodynamics, ventilator settings,
fluid balance, labs, procedures, and medication doses. Features are collected up until 36 hours post-surgery.
The modeling goal is sparse linear regression to predict the change in postoperative
serum creatinine after 60 hours. Specifically, we want to predict the ratio of postoperative
serum creatinine after 60 hours over the patient's baseline (pre-surgery) creatinine levels.
""".strip()

OUTCOME_NAME = "creatinine_ratio"

def infer_feature_type(series: pd.Series) -> str:
    """Light heuristic for the LLM (helps it reason about binary flags vs continuous)."""
    if pd.api.types.is_bool_dtype(series):
        return "binary"
    if pd.api.types.is_numeric_dtype(series):
        vals = series.dropna().unique()
        if len(vals) <= 3 and set(vals).issubset({0, 1}):
            return "binary"
        return "numeric"
    return "categorical_or_text"

def build_prompt_for_batch(feature_names: List[str]) -> str:
    # Keep the same lightweight metadata
    payload = []
    for i, name in enumerate(feature_names):
        ftype = infer_feature_type(df[name])
        payload.append({"id": i, "name": name, "type": ftype})

    features_json = json.dumps(payload, indent=2)

    prompt = f"""
You are a cardiothoracic ICU clinician and a biostatistician.

Background:
{DATASET_BACKGROUND}

Prediction target:
We are building a sparse linear regression model for {OUTCOME_NAME}, defined as the patient's
postoperative serum creatinine at Hour 60 divided by baseline (pre-surgery) creatinine.

Your task:
For each feature, assign an integer importance score from 1 to 5 for its usefulness in predicting
{OUTCOME_NAME} in post-cardiac surgery patients age 80 and older.

This is NOT a generic medical relevance task.
You must score features from the perspective of sparse model selection:
"If I had a limited feature budget, which variables from this batch would I keep because they add the
most unique predictive value for Hour 60 creatinine ratio?"

Core scoring strategy:
Evaluate features comparatively within this batch, not independently in isolation.

Think internally using these steps:
1. Identify which features are most likely to carry direct or near-direct information about future kidney dysfunction.
2. Penalize features that are mostly redundant with stronger versions of the same signal.
3. Favor features that would survive a sparse-model screening step because they add unique predictive value.
4. Distinguish direct renal signals from general illness-severity or clinician-behavior proxies.
5. Use the final 1-5 score to reflect selection priority for a sparse linear model.

Important principle:
A feature can be clinically relevant yet still receive only a moderate score if it is likely redundant,
indirect, poorly timed, or dominated by a stronger aggregation of the same underlying physiology.

Rules for Aggregated Vitals and Labs:
Many features end in _min, _max, _mean, and _measured and summarize a 12-hour post-surgery window.

You must explicitly reason as follows:

1. Dominance across aggregations:
If several versions of the same variable appear with different aggregations, give the highest score to the
aggregation that best preserves the clinically important renal signal, and reduce the others for redundancy.

2. Extremes (_min, _max):
These often capture acute insults better than averages.
Examples include hypotension, oliguria, peak creatinine, peak lactate, or extreme vasopressor requirement.
If an extreme is the most pathophysiologically informative version, it should dominate the family.

3. Means (_mean):
Means often smooth out short but important insults.
Score a _mean lower if it is likely less informative than the corresponding extreme.
Only keep a high score if sustained exposure itself is clinically important.

4. Measurement frequency (_measured):
This is mainly a proxy for monitoring intensity, clinician suspicion, or illness complexity.
Default these to low scores.
Only raise the score modestly if measurement frequency itself is meaningfully informative.
Do not treat _measured as a direct physiological biomarker.

Rule for Temporal Proximity (Time Epochs):
Features may come from three 12-hour epochs: 0_12h, 13_24h, and 25_36h.
The prediction target is at Hour 60.

Use these timing rules:
- 0_12h:
  Immediate post-surgical noise is common due to bypass, anesthesia, fluid shifts, and transient instability.
  Be conservative. Early abnormalities are often less reliable for Hour 60 prediction.

- 13_24h:
  This window is more informative because it reflects early trajectory, but still not the closest window.

- 25_36h:
  This is the closest available physiological window to Hour 60 and usually deserves the strongest weight
  if the feature is otherwise meaningful.

What to reward:
- Direct kidney function markers
- Strong precursors of renal hypoperfusion or evolving AKI
- Features that preserve the pathological signal in the right aggregation
- Features likely to survive sparse-model selection because they add unique value

What to penalize:
- Features that are mostly generic illness-severity markers
- Features that are mostly clinician behavior / monitoring proxies
- Redundant versions of a stronger signal
- Early transient post-op abnormalities that may not carry forward to Hour 60

Score interpretation:
Use the scores as sparse-model selection priority, not just medical importance.

Score 1:
Very weak candidate for inclusion.
Little direct relevance, mostly noise, routine care, weak proxy, or clearly dominated by better features.

Score 2:
Plausible but weak or indirect.
May reflect acuity or context, but unlikely to deserve selection in a sparse model unless stronger signals are absent.

Score 3:
Moderately useful.
Clinically relevant and may carry predictive information, but either indirect, somewhat redundant,
or not clearly top-tier for sparse selection.

Score 4:
Strong candidate for inclusion.
Clear predictive relevance to future renal dysfunction and likely to survive sparse-model screening.

Score 5:
Highest-priority feature in a sparse model.
Direct renal biomarker or especially strong near-term indicator of evolving AKI / renal hypoperfusion,
with high unique predictive value relative to other features in this batch.

Scoring guidance:
- Do not score features independently as if every relevant feature deserves a high score.
- Use the batch comparatively.
- Reserve 5 for true top-priority sparse-model candidates.
- Use 4 for strong keepers.
- Use 3 for useful but replaceable or partially redundant signals.
- Use 1-2 for weak, indirect, redundant, or proxy-heavy features.

Input features (JSON list):
{features_json}

Return a JSON object with key "scores" containing a list of objects.
Each object must include:
- id (same as input id)
- name (copy exactly)
- importance (integer 1..5)
- reason (1 to 2 concise sentences)

In each reason:
- briefly state what the feature represents clinically
- briefly justify the score in terms of sparse selection priority, timing, aggregation quality, redundancy, or proxy status
""".strip()

    return prompt


# ------------------------------------------------------------------------------
# Structured Output Schema
# ------------------------------------------------------------------------------
class ScoreItem(BaseModel):
    id: int
    name: str
    importance: int = Field(ge=1, le=5)
    reason: str

class ScoreBatch(BaseModel):
    scores: List[ScoreItem]


# ------------------------------------------------------------------------------
# LLM API Calls
# ------------------------------------------------------------------------------

client = OpenAI(api_key="API KEY HERE")

def call_llm_for_batch(prompt: str, model: str = "gpt-5.2") -> ScoreBatch:
    """
    Uses Responses API structured parsing, so you don't need to manually parse JSON text.
    """
    resp = client.responses.parse(
        model=model,
        input=[
            {"role": "system", "content": "You output only structured JSON that matches the schema."},
            {"role": "user", "content": prompt},
        ],
        text_format=ScoreBatch,
        temperature=0,
        max_output_tokens=2000,
    )
    return resp.output_parsed


def get_llm_importance_scores(
    feature_names: List[str],
    batch_size: Optional[int] = None,
    model: str = "gpt-5.2",
    sleep_s: float = 0.25,
    max_retries: int = 6,
    cache_path: str = "aki_llm_score_task.json",
) -> Dict[str, Dict]:
    """
    Returns:
      dict[feature_name] = {"importance": int(1..5), "reason": str}
    Caches incrementally so you can resume.
    """

    # default: ceil(sqrt(p)) like the paper’s heuristic
    if batch_size is None:
        batch_size = int(math.ceil(math.sqrt(len(feature_names))))

    # load cache if exists
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            results = json.load(f)
        print(f"[cache] loaded {len(results)} scores from {cache_path}")
    else:
        results = {}

    remaining = [f for f in feature_names if f not in results]
    print(f"Remaining features to score: {len(remaining)} (batch_size={batch_size})")

    for start in range(0, len(remaining), batch_size):
        batch = remaining[start : start + batch_size]
        prompt = build_prompt_for_batch(batch)

        # retry loop (simple exponential backoff)
        attempt = 0
        while True:
            try:
                parsed = call_llm_for_batch(prompt, model=model)
                # map back
                for item in parsed.scores:
                    results[item.name] = {
                        "importance": int(item.importance),
                        "reason": item.reason,
                    }
                # write cache
                with open(cache_path, "w") as f:
                    json.dump(results, f, indent=2)
                print(f"[ok] scored batch {start//batch_size + 1} / {math.ceil(len(remaining)/batch_size)}")
                break
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise RuntimeError(f"Failed after {max_retries} retries. Last error: {e}") from e
                backoff = (2 ** attempt) * 0.5
                print(f"[retry {attempt}] error={e} | sleeping {backoff:.1f}s")
                time.sleep(backoff)

        time.sleep(sleep_s)

    return results


# ------------------------------------------------------------------------------
# Run and Export
# ------------------------------------------------------------------------------

MODEL = "gpt-5.2"

raw = get_llm_importance_scores(
    FEATURE_COLS,
    batch_size=16,          # defaults to ceil(sqrt(p))
    model=MODEL,
    cache_path="aki_llm_score_task.json",
)

# Save raw weights (1..5)
raw_df = pd.DataFrame({
    "value": list(raw.keys()),
    "importance": [raw[k]["importance"] for k in raw],
    "reason": [raw[k]["reason"] for k in raw],
})
raw_df.to_csv("aki_weights_task.csv", index=False)

n_features = 1790
Remaining features to score: 1790 (batch_size=16)
[ok] scored batch 1 / 112
[ok] scored batch 2 / 112
[ok] scored batch 3 / 112
[ok] scored batch 4 / 112
[ok] scored batch 5 / 112
[ok] scored batch 6 / 112
[ok] scored batch 7 / 112
[ok] scored batch 8 / 112
[ok] scored batch 9 / 112
[ok] scored batch 10 / 112
[ok] scored batch 11 / 112
[ok] scored batch 12 / 112
[ok] scored batch 13 / 112
[ok] scored batch 14 / 112
[ok] scored batch 15 / 112
[ok] scored batch 16 / 112
[ok] scored batch 17 / 112
[ok] scored batch 18 / 112
[ok] scored batch 19 / 112
[ok] scored batch 20 / 112
[ok] scored batch 21 / 112
[ok] scored batch 22 / 112
[ok] scored batch 23 / 112
[ok] scored batch 24 / 112
[ok] scored batch 25 / 112
[ok] scored batch 26 / 112
[ok] scored batch 27 / 112
[ok] scored batch 28 / 112
[ok] scored batch 29 / 112
[ok] scored batch 30 / 112
[ok] scored batch 31 / 112
[ok] scored batch 32 / 112
[ok] scored batch 33 / 112
[ok] scored batch 34 / 112
[ok] scored batch 35 / 

In [4]:
from google.colab import files
files.download("aki_weights_task.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
df_new = pd.read_csv("aki_weights_task.csv")
df_new["importance"].value_counts()

,count
importance,
1,805
2,595
3,259
4,99
5,32
